In [ ]:
from datasets import load_dataset, concatenate_datasets
import random

In [ ]:
# Set random seed for reproducibility
random.seed(42)

In [ ]:
def get_subset(dataset_name, lang_label, n=500):

    dataset = load_dataset(dataset_name, split="train", streaming=True)

    subset = []
    for i, example in enumerate(dataset):
        example["lang"] = lang_label
        subset.append(example)

        if i + 1 == n:
            break

    return subset

In [ ]:
#Take first 500 samples
tamil_subset = get_subset("SPRINGLab/IndicVoices-R_Tamil", 0)
hindi_subset = get_subset("SPRINGLab/IndicVoices-R_Hindi", 1)
bengali_subset = get_subset("SPRINGLab/IndicVoices-R_Bengali", 2)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Combine datasets
combined_data = tamil_subset + hindi_subset + bengali_subset

In [ ]:
# Shuffle dataset with seed 42
random.shuffle(combined_data)

In [ ]:
print(len(combined_data))  # 1500
print(combined_data[0])

1500
{'text': 'শাড়ি তৈরি করে তাঁতের শাড়ি তৈরি করে ন্যায্য মূল্যে বিক্রি করে এবং আমাদের এখান থেকে অনেক মানুষ সেখানে শাড়ি কিনতে যার ফলে ন্যায্য মূল্য না পাওয়ার জন্য তাদের খুব সমস্যা হয়', 'lang': 2, 'samples': 477030, 'verbatim': 'শাড়ি তৈরি করে তাঁতের শাড়ি তৈরি করে ন্যায্য মূল্যে বিক্রি করে এবং আমাদের এখান থেকে অনেক মানুষ সেখানে শাড়ি কিনতে যার ফলে ন্যায্য মূল্য না পাওয়ার জন্য তাদের খুব সমস্যা হয়', 'normalized': 'শাড়ি তৈরি করে তাঁতের শাড়ি তৈরি করে ন্যায্য মূল্যে বিক্রি করে এবং আমাদের এখান থেকে অনেক মানুষ সেখানে শাড়ি কিনতে যার ফলে ন্যায্য মূল্য না পাওয়ার জন্য তাদের খুব সমস্যা হয়', 'speaker_id': 'S4258014500311887', 'scenario': 0, 'task_name': 'District Specific ', 'gender': 0, 'age_group': 2, 'job_type': 2, 'qualification': 3, 'area': 0, 'district': 'Nadia', 'state': 0, 'occupation': 'House wife', 'utterance_pitch_mean': 215.095, 'utterance_pitch_std': 14.8906975, 'snr': 55.088432312, 'c50': 59.8574905396, 'speaking_rate': 15.4386316319, 'cer': 'tensor(0.0051)', 'duration': 10.8170208333, '

In [ ]:
example = tamil_subset[0]
print(example["audio"]["sampling_rate"])

48000


In [ ]:
example = hindi_subset[0]
print(example["audio"]["sampling_rate"])

48000


In [ ]:
example = bengali_subset[0]
print(example["audio"]["sampling_rate"])

48000


In [ ]:
example = hindi_subset[15]
print(example["speaker_id"])

S4259027400309283


In [ ]:
import torch
import torchaudio
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from datasets import load_dataset

In [ ]:
model_name = "facebook/wav2vec2-base-960h"

processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2Model.from_pretrained(model_name)

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
model.config.output_hidden_states = True

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Wav2Vec2Model(
  (feature_extractor): Wav2Vec2FeatureEncoder(
    (conv_layers): ModuleList(
      (0): Wav2Vec2GroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): Wav2Vec2FeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): Wav2Vec2Encoder(
    (pos_conv_embed): Wav2Vec2PositionalConvEmbedding(
  

In [ ]:
def extract_features(batch_audio):
    inputs = processor(
        batch_audio,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    # Move to device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    hidden_states = outputs.hidden_states

    # Extract layers
    layer_9 = hidden_states[7]
    layer_11 = hidden_states[12]

    # Average layers
    combined = (layer_9 + layer_11) / 2

    # Simple mean pooling (no masking)
    features = combined.mean(dim=1)

    return features

In [ ]:
def process_in_batches(data, batch_size=4):
    all_features = []
    all_labels = []

    for i in range(0, len(data), batch_size):
        print("processing..." + str(i/batch_size))
        batch = data[i:i+batch_size]

        # Extract audio arrays
        audio_list = [x["audio"]["array"] for x in batch]
        labels = [x["lang"] for x in batch]

        # Extract features
        features = extract_features(audio_list)

        all_features.append(features)
        all_labels.extend(labels)

    # Combine all batches
    all_features = torch.cat(all_features, dim=0)

    return all_features, torch.tensor(all_labels)

In [ ]:
features, labels = process_in_batches(combined_data, batch_size=4)
print(features.shape)  # Expected: (1500, 768)
print(labels.shape)    # Expected: (1500,)

processing...0.0
processing...1.0
processing...2.0
processing...3.0
processing...4.0
processing...5.0
processing...6.0
processing...7.0
processing...8.0
processing...9.0
processing...10.0
processing...11.0
processing...12.0
processing...13.0
processing...14.0
processing...15.0
processing...16.0
processing...17.0
processing...18.0
processing...19.0
processing...20.0
processing...21.0
processing...22.0
processing...23.0
processing...24.0
processing...25.0
processing...26.0
processing...27.0
processing...28.0
processing...29.0
processing...30.0
processing...31.0
processing...32.0
processing...33.0
processing...34.0
processing...35.0
processing...36.0
processing...37.0
processing...38.0
processing...39.0
processing...40.0
processing...41.0
processing...42.0
processing...43.0
processing...44.0
processing...45.0
processing...46.0
processing...47.0
processing...48.0
processing...49.0
processing...50.0
processing...51.0
processing...52.0
processing...53.0
processing...54.0
processing...55.0
pr

In [ ]:
first_value = features[0][0].item()
print(round(first_value, 4))

0.0199


In [ ]:
tamil_subset[24]["gender"]

0

In [ ]:
import numpy as np

X = features.detach().cpu().numpy()   # shape → (1500, 768)
y = np.array([item["lang"] for item in combined_data])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_final, y_train_final)

LogisticRegression(max_iter=1000, random_state=42)

In [ ]:
from sklearn.metrics import accuracy_score

y_val_pred = model.predict(X_val)

val_accuracy = accuracy_score(y_val, y_val_pred) * 100
print(f"{val_accuracy:.2f}")

70.00


In [ ]:
from sklearn.metrics import accuracy_score

y_test_pred = model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_test_pred) * 100
print(f"{test_accuracy:.4f}")

66.6667


In [ ]:
from transformers import Wav2Vec2Model

model_name = "facebook/wav2vec2-base-960h"
model = Wav2Vec2Model.from_pretrained(model_name)

# Total parameters
total_params = sum(p.numel() for p in model.parameters())

# Convert to millions
total_params_millions = total_params / 1e6

print(f"Total parameters: {total_params_millions:.2f} million")

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters: 94.37 million
